In [11]:
from sklearn.model_selection import train_test_split
from data import load_data, get_random_state
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.inspection import permutation_importance
from pygam import LogisticGAM, s
import pandas as pd


In [10]:
df = load_data()
RANDOM_STATE = get_random_state()

X = df.drop(columns=["target"])
y = df["target"]

categorical_features = ["sex", "cp", "fbs", "restecg", "exang", "slope", "ca", "thal"]
numeric_features = [c for c in X.columns if c not in categorical_features]

# For GAM, keep a smaller numeric subset for stable smooth curves
gam_features = ["age", "trestbps", "chol", "thalach", "oldpeak"]
X_gam = df[gam_features].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

Xg_train, Xg_test, yg_train, yg_test = train_test_split(
    X_gam, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

In [12]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median"))
        ]), numeric_features),
        ("cat", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), categorical_features),
    ]
)

rf_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(
            n_estimators=300,
            max_depth=6,
            min_samples_leaf=5,
            random_state=RANDOM_STATE
        ))
    ]
)

rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)
rf_prob = rf_model.predict_proba(X_test)[:, 1]

rf_accuracy = accuracy_score(y_test, rf_pred)
rf_auc = roc_auc_score(y_test, rf_prob)

# Permutation importance on original features
perm = permutation_importance(
    rf_model, X_test, y_test,
    n_repeats=10,
    random_state=RANDOM_STATE,
    scoring="roc_auc"
)
rf_importance_df = pd.DataFrame({
    "feature": X.columns,
    "importance": perm.importances_mean
}).sort_values("importance", ascending=False)

/Users/nicoleganin/Documents/Documenten - MacBook Pro van Nicole/VisAnalytics/visual_analytics/.venv/lib/python3.13/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: ['ca' 'thal']. At least one non-missing value is needed for imputation with strategy='most_frequent'.
  warnings.warn(
/Users/nicoleganin/Documents/Documenten - MacBook Pro van Nicole/VisAnalytics/visual_analytics/.venv/lib/python3.13/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: ['ca' 'thal']. At least one non-missing value is needed for imputation with strategy='most_frequent'.
  warnings.warn(
/Users/nicoleganin/Documents/Documenten - MacBook Pro van Nicole/VisAnalytics/visual_analytics/.venv/lib/python3.13/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: ['ca' 'thal']. At least one non-missing value is needed for imputation with strategy='most_frequent'.
  w

In [ ]:
# Patient Explorer 
    dbc.Card(
        [
            dbc.CardHeader(html.H2("Patient Explorer", className="mb-0"), className='bg-light'),
            dbc.CardBody([
                dbc.Row([
                    dbc.Col([
                        html.Label("Select dimensions (at least 3)"),
                        dcc.Checklist(
                            id='pcp-feature-selector',
                            options=[{'label': label, 'value': key} for key, label in FEATURE_OPTIONS.items()],
                            value=['age', 'trestbps', 'chol', 'thalach'],
                            style={'display': 'none'},
                        ),
                        html.Div(id='pcp-selected-badges', className='mb-3 mt-2'),
                        html.Div("(Click badges to select/unselect)", className='text-muted mb-3 mt-2'),

                        html.Label("Age range"),
                        dcc.RangeSlider(
                            id='age-range-slider',
                            min=int(data['age'].min()),
                            max=int(data['age'].max()),
                            step=1,
                            value=[int(data['age'].min()), int(data['age'].max())],
                            marks={i: str(i) for i in range(int(data['age'].min()), int(data['age'].max())+1, 5)},
                            tooltip={'placement': 'bottom', 'always_visible': False},
                            className='mb-4'
                        ),

                        html.Label("Target"),
                        dcc.Checklist(
                            id='target-filter',
                            options=[
                                {'label': 'No Disease', 'value': 0},
                                {'label': 'Disease', 'value': 1},
                            ],
                            value=[0, 1],
                            inline=True,
                        ),

                    ], width=4),

                    dbc.Col(
                        dcc.Graph(id='parallel-coords-chart', style={'height': '520px'}),
                        width=8
                    )
                ]),

                html.H4("Filtered Patient Table", className='mt-4'),
                dash_table.DataTable(
                    id='patient-data-table',
                    columns=[{"name": c, "id": c} for c in data.columns],
                    data=data.to_dict('records'),
                    page_size=10,
                    style_table={'overflowX': 'auto'},
                    style_cell={'textAlign': 'left', 'minWidth': '100px', 'width': '150px', 'maxWidth': '240px'},
                ),
            ])
        ],
        color='secondary',
        outline=True,
        className='mb-4'
    
    ),